In [1]:
import os
#os.chdir("C:\\Users\\Le Tian\\Desktop\\Ensemble modeling\\comp-401")
os.chdir("/Users/gaoletian/Desktop/Ensemble modeling/comp-401")
os.getcwd()

'/Users/gaoletian/Desktop/Ensemble modeling/comp-401'

In [3]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from multi_omics_integration.processing import load_resize
from multi_omics_integration.func import estimators, estimator_parameters

In [4]:
datasets = {
    "cnv": "../Datasets/kipan/subtyping/CNV.csv"
}
labels = "../Datasets/kipan/subtyping/Clinical_enc.csv"
target_name = "histological_type_enc"

merged_X, X_modalities, y = load_resize(datasets, labels, target_name)

In [5]:
import scanpy as sc
import pandas as pd

# Load data
cnv = pd.read_csv(datasets['cnv'], index_col=0)

# CNV preprocessing

# OPTIONNAL
n_cnv = cnv.clip(lower=-2, upper=2)
n2_cnv = n_cnv.map(lambda x: 1 if x > 0.5 else (-1 if x < -0.5 else 0))

from sklearn.preprocessing import StandardScaler
from pandas import DataFrame

scaler = StandardScaler()
cnv_df = DataFrame(scaler.fit_transform(n2_cnv), columns=cnv.columns)
print(cnv_df.shape)

print("Processed CNV shape:", cnv_df.shape)

(736, 24776)
Processed CNV shape: (736, 24776)


In [7]:
import pandas as pd

# Convert y to a pandas Series with same index as rna_processed
y = pd.Series(y, index=cnv_df.index)

# Now reassign
merged_X = cnv_df.loc[y.index]

print(" Merged shape:", merged_X.shape)
print(" Target distribution:\n", y.value_counts())


 Merged shape: (736, 24776)
 Target distribution:
 1    467
2    206
0     63
Name: count, dtype: int64


In [9]:
results = []

for name, model in estimators:
    if name not in estimator_parameters:
        print(f"Skipping '{name}' (no param grid defined)")
        continue

    print(f"\nTuning model: {name}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])

    param_grid = {f"clf__{k}": v for k, v in estimator_parameters[name].items()}

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        verbose=1,
        n_jobs=-1
    )

    try:
        grid.fit(merged_X, y)
        results.append({
            "model": name,
            "best_score": grid.best_score_,
            "best_params": grid.best_params_
        })
        print(f" Best score for {name}: {grid.best_score_:.4f}")
        print(f" Best parameters for {name}: {grid.best_params_}")
    except Exception as e:
        print(f" Failed to fit {name}: {e}")


Tuning model: logistic
Fitting 5 folds for each of 5 candidates, totalling 25 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in ve

 Best score for logistic: 0.8573
 Best parameters for logistic: {'clf__C': 0.01}

Tuning model: random_forest
Fitting 5 folds for each of 16 candidates, totalling 80 fits
 Best score for random_forest: 0.8750
 Best parameters for random_forest: {'clf__max_depth': 7, 'clf__n_estimators': 100}

Tuning model: deep_nn
Fitting 5 folds for each of 864 candidates, totalling 4320 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="best_score", ascending=False)
print("\nSummary of Results:")
display(results_df)